# 08 — Dockerising the Credit Card Underwriting API

This notebook explains how and why we containerise the application using Docker. It is meant to be read top-to-bottom — each section explains **what** Docker is, **why** we use it, and **how** every line of our `Dockerfile` works.

---

## What is Docker?

Docker is a tool that packages your application and everything it needs to run — Python version, libraries, code, config — into a single portable unit called a **container**.

Think of it like this:

| Without Docker | With Docker |
|---|---|
| "It works on my machine" | Runs identically everywhere |
| Manually install Python, pip, dependencies on every machine | All dependencies baked into the image |
| Different Python versions cause conflicts | Each app has its own isolated environment |
| Hard to deploy — server needs to match your dev setup | Just run the container on any machine |

---

## Core Concepts

Before looking at code, here are the four concepts you need to understand:

### Image
A **read-only blueprint** for a container. It contains the OS, Python, your dependencies, and your code. You build an image once with `docker build`.

### Container
A **running instance** of an image. Like a class vs an object in Python — the image is the class, the container is the object. You can run many containers from one image.

### Dockerfile
A **text file of instructions** that tells Docker how to build your image. Each line is a step (install Python, copy files, run pip install, etc).

### Volume
A **persistent storage link** between the container and your local machine. By default, any files written inside a container are lost when it stops. Volumes let you keep data (like our SQLite database) alive between restarts.

---

## Why does this project need Docker?

Our app has several moving parts that need to be set up correctly:

- Python 3.13
- All packages from `requirements.txt` (XGBoost, FastAPI, SQLAlchemy, etc.)
- The trained model file (`models/xgb_default.pkl`)
- A `SECRET_KEY` environment variable for JWT signing
- A writable directory for the SQLite database (`instance/`)

Without Docker, anyone running this project has to manually install all of this and hope their environment matches ours. With Docker, one command gives them a running app.

---

## The Dockerfile

Here is the full `Dockerfile` we created at the root of the project, with every line explained:

```dockerfile
FROM python:3.13-slim
```

**`FROM`** sets the base image — the starting point Docker builds on top of.

- `python:3.13-slim` is an official Docker image that comes with Python 3.13 pre-installed on a minimal Linux OS.
- `slim` means it strips out unnecessary tools (compilers, docs, etc.) to keep the image small. Our app only needs Python — we don't need the full OS.
- This matches our local Python version so behaviour is consistent.

---

```dockerfile
WORKDIR /app
```

**`WORKDIR`** sets the working directory inside the container. All subsequent commands run from `/app`, and all file paths are relative to it.

Without this, files would end up scattered in the root `/` directory, which is messy and can cause permission issues.

---

```dockerfile
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
```

**`COPY`** copies a file from your local machine into the container.

**`RUN`** executes a shell command during the build.

These two lines are intentionally placed **before copying the rest of the code**. This is a Docker best practice called **layer caching**:

- Docker builds images in layers — one per instruction.
- If a layer hasn't changed, Docker reuses the cached version instead of rebuilding it.
- Dependencies rarely change, but code changes constantly.
- By installing dependencies first, every subsequent `docker build` after a code change skips the slow `pip install` step and uses the cached layer.

`--no-cache-dir` tells pip not to store downloaded packages in a cache directory inside the container, keeping the image smaller.

---

```dockerfile
COPY api/ ./api/
COPY src/ ./src/
COPY models/ ./models/
```

These lines copy our application code into the container:

- `api/` — the FastAPI app (routes, schemas, templates, auth, etc.)
- `src/` — the data loading and feature engineering modules
- `models/` — the trained XGBoost model file (`xgb_default.pkl`)

The model is baked into the image because it is read-only — it doesn't change at runtime. The SQLite database is **not** copied in (it's handled by a volume — see below).

---

```dockerfile
RUN mkdir -p instance
```

Creates the `instance/` directory inside the container. This is where `api/database.py` writes the SQLite file (`instance/predictions.db`).

We create it here so it exists when the container starts. When running with a volume (`-v`), Docker mounts over this directory — but it must exist first.

---

```dockerfile
EXPOSE 8000
```

Documents that the container listens on port 8000. This is metadata — it doesn't actually open the port. The port is opened when you run the container with `-p 8000:8000`.

It is good practice to include it so anyone reading the Dockerfile knows which port the app uses.

---

```dockerfile
CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

**`CMD`** defines the default command to run when the container starts.

- `uvicorn` — the ASGI server that runs our FastAPI app
- `api.main:app` — the Python module path: `api/main.py`, the `app` object
- `--host 0.0.0.0` — listen on all network interfaces inside the container, not just `localhost`. Without this, the app would only be reachable from inside the container itself.
- `--port 8000` — the port the server listens on inside the container

In [ ]:
# The complete Dockerfile — for reference

DOCKERFILE = """
FROM python:3.13-slim

WORKDIR /app

# Install dependencies first (cached layer)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code and model
COPY api/ ./api/
COPY src/ ./src/
COPY models/ ./models/

# instance/ is the SQLite data directory — mount a volume here in production
RUN mkdir -p instance

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print(DOCKERFILE)

---

## The .dockerignore File

Just like `.gitignore` tells git which files to skip, `.dockerignore` tells Docker which files to exclude from the build context — the set of files Docker can see when building the image.

Here is our `.dockerignore`:

```
.cc_venv/
__pycache__/
*.pyc
*.pyo
.git/
data/
notebooks/
reports/
tests/
instance/
*.db
*.ipynb
.env
```

**Why does this matter?**

When you run `docker build`, Docker sends your entire project folder to the Docker daemon. Without `.dockerignore`, this includes your virtual environment (hundreds of MB), your raw data CSV, git history, notebooks, and more — none of which belong in the image.

Line by line:

| Entry | Why excluded |
|---|---|
| `.cc_venv/` | Virtual environment — Docker installs its own from `requirements.txt` |
| `__pycache__/`, `*.pyc`, `*.pyo` | Python bytecode — generated fresh inside the container |
| `.git/` | Git history — not needed at runtime, adds size |
| `data/` | Raw CSV files — not needed by the API at runtime |
| `notebooks/` | Jupyter notebooks — development artefacts, not runtime code |
| `reports/` | Plots and outputs — not needed at runtime |
| `tests/` | Test files — not run inside the container |
| `instance/` | Local SQLite DB — the container gets a fresh DB via a volume |
| `*.db` | Any stray database files |
| `*.ipynb` | Any stray notebook files |
| `.env` | Local environment file — secrets must never be baked into images |

---

## Building the Image

```bash
docker build -t cc-underwriting .
```

Breaking this down:

- `docker build` — tells Docker to build an image using the `Dockerfile` in the current directory
- `-t cc-underwriting` — tags (names) the image `cc-underwriting` so you can refer to it by name instead of a hash
- `.` — the build context: the current directory. Docker looks for a `Dockerfile` here and uses `.dockerignore` to decide what to include

**What happens during the build?**

Docker executes each instruction in the `Dockerfile` in order, creating a new layer for each one:

```
Step 1/8: FROM python:3.13-slim          ← download base image
Step 2/8: WORKDIR /app                   ← set working directory
Step 3/8: COPY requirements.txt .        ← copy requirements
Step 4/8: RUN pip install ...            ← install dependencies (slow first time)
Step 5/8: COPY api/ ./api/              ← copy app code
Step 6/8: COPY src/ ./src/              ← copy src modules
Step 7/8: COPY models/ ./models/        ← copy trained model
Step 8/8: RUN mkdir -p instance          ← create DB directory
```

The second time you build (after a code change), Docker skips steps 1–4 because nothing changed — it uses the cached layers. Only the `COPY` steps after the change re-run.

---

## Running the Container

```bash
docker run -p 8000:8000 \
  -e SECRET_KEY=$(python3 -c 'import secrets; print(secrets.token_hex(32))') \
  -v $(pwd)/instance:/app/instance \
  cc-underwriting
```

Each flag explained:

### `-p 8000:8000` — Port mapping

Format: `-p <host_port>:<container_port>`

The container has its own isolated network. Port 8000 inside the container is not automatically accessible from your machine. This flag creates a bridge:

```
Your browser → localhost:8000 → Docker → container:8000 → uvicorn
```

Both numbers are 8000 here, but they don't have to match. `-p 9000:8000` would make the app available at `localhost:9000` while uvicorn still runs on 8000 inside.

---

### `-e SECRET_KEY=...` — Environment variable

Format: `-e KEY=value`

Our app reads `SECRET_KEY` from the environment in `api/security.py`:

```python
SECRET_KEY: str = os.environ.get("SECRET_KEY", "")
```

This key signs every JWT token. We generate a fresh cryptographically random value at run time using Python's `secrets` module:

```bash
python3 -c 'import secrets; print(secrets.token_hex(32))'
# → e.g. 4a7c9f2b1d8e3a6f...  (64 hex chars = 32 bytes = 256 bits)
```

**Important:** Use the same `SECRET_KEY` every time you restart the container. If you generate a new one each time, all existing JWT tokens become invalid and logged-in users are immediately signed out.

In production you would store the key in a secrets manager and pass it in via your deployment platform rather than generating it on the command line.

---

### `-v $(pwd)/instance:/app/instance` — Volume mount

Format: `-v <host_path>:<container_path>`

By default, any file written inside a container is lost when the container stops. Our SQLite database lives at `instance/predictions.db` inside the container. Without a volume, every restart wipes the entire prediction history.

This flag mounts your local `instance/` folder into the container at `/app/instance`. Writes from the container go directly to your local disk:

```
Container writes to /app/instance/predictions.db
         ↕  (same file)
Your Mac at ./instance/predictions.db
```

The database now persists across restarts, rebuilds, and even new versions of the image.

---

### `cc-underwriting` — Image name

The name we gave the image with `-t` during `docker build`. This tells Docker which image to start a container from.

---

## How the App Starts Inside the Container

When the container starts, Docker runs:

```bash
uvicorn api.main:app --host 0.0.0.0 --port 8000
```

`api/main.py` runs its bootstrap code immediately:

```python
# 1. Create the SQLite tables if they don't exist
Base.metadata.create_all(bind=engine)

# 2. Load the trained XGBoost model from disk
_model = joblib.load(MODEL_PATH)   # models/xgb_default.pkl

# 3. Read feature names from the model
FEATURE_NAMES = _model.get_booster().feature_names
```

If any of these fail — for example if `models/xgb_default.pkl` was not copied into the image — the container exits immediately with an error. This is intentional: a misconfigured container fails loudly rather than silently serving broken responses.

---

## Useful Docker Commands

```bash
# See all running containers
docker ps

# See all images on your machine
docker images

# Stop a running container (Ctrl+C also works if it's in the foreground)
docker stop <container_id>

# Remove an image (to force a clean rebuild)
docker rmi cc-underwriting

# View logs from a running container
docker logs <container_id>

# Open a shell inside a running container (useful for debugging)
docker exec -it <container_id> bash
```

---

## Summary

| Concept | What it does in this project |
|---|---|
| `FROM python:3.13-slim` | Starts from an official Python image matching our dev environment |
| `WORKDIR /app` | Sets all paths relative to `/app` inside the container |
| `COPY requirements.txt` + `RUN pip install` | Installs dependencies in a cached layer — fast rebuilds after code changes |
| `COPY api/ src/ models/` | Bundles our code and trained model into the image |
| `RUN mkdir -p instance` | Ensures the SQLite directory exists before the app starts |
| `EXPOSE 8000` | Documents the port the app listens on |
| `CMD ["uvicorn", ...]` | Starts the API server when the container launches |
| `.dockerignore` | Excludes venv, data, notebooks, git history from the build |
| `-p 8000:8000` | Maps the container's port 8000 to your machine's port 8000 |
| `-e SECRET_KEY=...` | Injects the JWT signing secret without baking it into the image |
| `-v $(pwd)/instance:/app/instance` | Mounts a local folder so the SQLite database survives restarts |

### The full workflow

```
1.  docker build -t cc-underwriting .      ← build the image (once, then only after changes)

2.  docker run -p 8000:8000 \
      -e SECRET_KEY=<your_key> \
      -v $(pwd)/instance:/app/instance \
      cc-underwriting                       ← start the container

3.  Open http://localhost:8000             ← use the app
```